# Strict Retraining Candidate Queue V1

This notebook rebuilds the no-promotion retraining queue from the current strict walk-forward retention evidence. It keeps SOXLBacktest as a qlib-aligned buy/sell reference baseline and identifies the next actor-training candidates that still need official walk-forward proof before retention.

Governance: final actor retention requires all required walk-forward windows, worst WF max drawdown below 20%, return alignment with the current best model, and a passing final walk-forward retention contract.


In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'AGENTS.md').exists():
            return candidate
    raise RuntimeError(f'repo root not found from {start}')

ROOT = find_repo_root(Path.cwd())
CARD = ROOT / 'runtime/model_selection/card/strict_retraining_candidate_queue_v1'
CARD.mkdir(parents=True, exist_ok=True)
print(ROOT)


/Users/rziiiii/dev/workspace


## Rebuild Artifacts

Run the repo-owned script so the notebook reproduces `summary.json`, `candidate_queue.csv`, `required_gates.csv`, `source_manifest.csv`, and the Markdown sidecar.


In [2]:
cmd = [sys.executable, 'scripts/review_strict_retraining_candidate_queue.py']
completed = subprocess.run(cmd, cwd=ROOT, check=True, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)


strict_retraining_candidate_queue decision=strict_retraining_candidate_queue_ready_no_promotion top=mode_conditioned_rebound_reward_train_v2 queue=14 output=/Users/rziiiii/dev/workspace/runtime/model_selection/card/strict_retraining_candidate_queue_v1



## Queue And Gates

The first trained actor is the next practical retraining target. Reference baselines and screen-only entries are explicitly not promotable.


In [3]:
summary_payload = json.loads((CARD / 'summary.json').read_text())
summary = summary_payload.get('summary', summary_payload)
queue = pd.read_csv(CARD / 'candidate_queue.csv')
gates = pd.read_csv(CARD / 'required_gates.csv')
sources = pd.read_csv(CARD / 'source_manifest.csv')
print(json.dumps(summary, indent=2, sort_keys=True))
display_cols = [
    'queue_rank', 'candidate_id', 'candidate_type', 'queue_role',
    'ytd_return_pct', 'worst_wf_max_drawdown_pct', 'wf_drawdown_gap_pct',
    'return_gap_pct', 'missing_wf_window_count', 'primary_blocker',
]
queue[display_cols].head(8)


{
  "actor_retention_allowed_count": 0,
  "candidate_filter_actor_retention_allowed_count": 0,
  "current_best_return_pct": 224.361539,
  "decision": "strict_retraining_candidate_queue_ready_no_promotion",
  "final_retained_count": 0,
  "max_walkforward_drawdown_pct": 20.0,
  "next_action": "short_foldlocal_bc_or_ppo_training_screen_with_drawdown_drift_reward_and_official_wf_gate",
  "promotion_allowed_count": 0,
  "queue_count": 14,
  "reference_baseline": "soxl_backtest_policy_reimplementation_v1",
  "reference_return_pct": 146.043798,
  "required_best_return_pct": 213.143462,
  "required_wf_windows": [
    "wf_2020",
    "wf_2022",
    "wf_2023_2024"
  ],
  "sidecars": {
    "candidate_queue_csv": "runtime/model_selection/card/strict_retraining_candidate_queue_v1/candidate_queue.csv",
    "markdown": "runtime/model_selection/card/strict_retraining_candidate_queue_v1/strict_retraining_candidate_queue.md",
    "required_gates_csv": "runtime/model_selection/card/strict_retraining_candi

,queue_rank,candidate_id,candidate_type,queue_role,ytd_return_pct,worst_wf_max_drawdown_pct,wf_drawdown_gap_pct,return_gap_pct,missing_wf_window_count,primary_blocker
0,1,mode_conditioned_rebound_reward_train_v2,trained_actor,train_drawdown_drift_and_return_alignment,177.573569,21.831720,1.831720,35.569893,0.0,drawdown_and_return_gap
1,2,drawdown_rebound_reward_train_v1,trained_actor,train_drawdown_drift_and_return_alignment,172.832548,23.546830,3.546830,40.310914,0.0,drawdown_and_return_gap
2,3,state_conditioned_drawdown_rebound_reward_trai...,trained_actor,train_drawdown_drift_and_return_alignment,196.505054,27.608986,7.608986,16.638408,0.0,drawdown_and_return_gap
3,4,balanced_rule_residual_imitation_v4,trained_actor,train_drawdown_drift_and_return_alignment,145.031235,35.710864,15.710864,68.112227,0.0,drawdown_and_return_gap
4,5,balanced_rule_residual_v19,trained_actor,train_drawdown_drift_and_return_alignment,173.299227,46.653957,26.653957,39.844235,0.0,drawdown_and_return_gap
5,6,balanced_rule_residual_v19_reg,trained_actor,train_drawdown_drift_and_return_alignment,141.992184,46.171637,26.171637,71.151278,0.0,drawdown_and_return_gap
6,7,balanced_rule_residual_v19_reg_light,trained_actor,train_drawdown_drift_and_return_alignment,164.253638,51.596776,31.596776,48.889824,0.0,drawdown_and_return_gap
7,8,balanced_rule_residual_v22,trained_actor,train_drawdown_drift_and_return_alignment,116.882644,54.665610,34.665610,96.260818,0.0,drawdown_and_return_gap


In [4]:
gates


,gate,required,current,status
0,final_walkforward_retention_contract,True,0.000000,blocked
1,strict_walkforward_profile_count,1,0.000000,blocked
2,max_walkforward_drawdown_pct,20.0,21.831720,blocked
3,required_best_return_pct,213.143462,177.573569,blocked


In [5]:
sources


,source_label,path,exists,role
0,strict_walkforward_gate_summary,runtime/model_selection/card/model_retention_w...,True,strict WF gate summary and thresholds
1,strict_walkforward_gate_rows,runtime/model_selection/card/model_retention_w...,True,per-profile current gate failures and gaps
2,final_walkforward_retention_contract,runtime/model_selection/card/final_walkforward...,True,final actor-retention contract
3,strict_candidate_retention_filter,runtime/model_selection/card/strict_candidate_...,True,no-promotion actor retention filter
4,soxlbacktest_qlib_walkforward_baseline,runtime/model_selection/card/soxlbacktest_qlib...,True,SOXLBacktest qlib buy/sell reference baseline
5,final_walkforward_retraining_handoff,runtime/model_selection/card/final_walkforward...,True,screen-only next training handoff


## Conclusion

- Decision: `strict_retraining_candidate_queue_ready_no_promotion`.
- Current final retained actor count remains `0`; this notebook does not promote any actor.
- Top retraining target: `mode_conditioned_rebound_reward_train_v2`, because it has all required WF windows and is closest to the 20% drawdown gate, but it still needs both drawdown improvement (`21.83172%` worst WF MaxDD, gap `1.83172pp`) and return alignment (`177.573569%` YTD, gap `35.569893pp` to required best alignment).
- `smoke` remains a return-aligned diagnostic actor only; it must complete `wf_2020`, `wf_2022`, and `wf_2023_2024` before actor retention can be considered.
- `soxl_backtest_policy_reimplementation_v1` remains a qlib-aligned buy/sell reference baseline only, not a runnable actor.
- Next action: run a short fold-local drawdown-drift / trend-participation screen, then require the official strict WF gate and final walk-forward retention contract before retaining any model.
